[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prompt-engineering-certified/notebooks/day-03-chain-of-thought.ipynb#scrollTo=aa1b2c3d)

---
# Day 3 · Chain-of-Thought Reasoning
**certified-journeys / prompt-engineering-certified** · Day 3 · Advanced Techniques

> **Goal for today:** Implement zero-shot CoT, few-shot CoT with worked examples, and self-consistency majority voting; measure each approach on math word problems; and document a concrete case where CoT produces a confident but wrong answer.


In [ ]:
%pip install -q openai


## Step 1 · Setup and problem bank

We need a set of math word problems with known correct answers — our ground truth for measuring  
whether CoT actually improves accuracy over direct answering.

We'll use three problems of increasing complexity:
1. Simple arithmetic (easy baseline)
2. Multi-step rate problem (where direct answering often fails)
3. Combinatorics (where CoT helps most)


In [ ]:
import os
import re
import json
from collections import Counter
from typing import Optional

# ── Mock layer ───────────────────────────────────────────────────────────────
MOCK = os.environ.get("OPENAI_API_KEY") is None

def chat(
    prompt: str,
    mock_response: str = "42",
    model: str = "gpt-4o-mini",
    temperature: float = 0.0,
) -> str:
    """Call OpenAI or return a mock response."""
    if MOCK:
        return mock_response
    from openai import OpenAI
    client = OpenAI()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return response.choices[0].message.content

print(f"Running in {'MOCK' if MOCK else 'LIVE'} mode.")

# ── Problem bank ─────────────────────────────────────────────────────────────
PROBLEMS = [
    {
        "id": "P1",
        "text": (
            "A baker makes 48 cookies. She sells 3/4 of them in the morning "
            "and 6 more in the afternoon. How many cookies does she have left?"
        ),
        "answer": 6,
        "difficulty": "easy",
    },
    {
        "id": "P2",
        "text": (
            "Train A leaves City X at 9 AM travelling at 60 mph. "
            "Train B leaves City Y (300 miles away) at 10 AM travelling toward City X at 90 mph. "
            "At what time do the trains meet? Give the answer as a time (e.g. 11:00 AM)."
        ),
        "answer": "11:00 AM",
        "difficulty": "medium",
    },
    {
        "id": "P3",
        "text": (
            "A committee of 3 people is chosen from a group of 5 men and 4 women. "
            "How many committees contain at least 1 woman?"
        ),
        "answer": 74,
        "difficulty": "hard",
    },
]

def extract_final_number(text: str) -> Optional[str]:
    """Extract the last number mentioned in a response for comparison."""
    numbers = re.findall(r"\b\d+(?:\.\d+)?\b", text)
    return numbers[-1] if numbers else None

for p in PROBLEMS:
    print(f"{p['id']} ({p['difficulty']}): {p['text'][:70]}...  [answer: {p['answer']}]")


## Step 2 · Direct Answering Baseline (no CoT)

Before measuring CoT's benefit, we need a baseline: how does the model perform when asked  
to answer directly without reasoning?

The **direct prompt** deliberately discourages reasoning by asking for a bare answer —  
this isolates CoT's contribution in the next step.


In [ ]:
# ── Direct-answer prompt (no CoT) ─────────────────────────────────────────────
def direct_prompt(problem_text: str) -> str:
    return (
        f"{problem_text}\n\n"
        "Answer with a single number or time only. No working, no explanation."
    )

# Mock: direct answers — model gets P1 right, fails P2 (common error), fails P3
DIRECT_MOCK = {
    "P1": "6",         # correct
    "P2": "11:30 AM",  # wrong (common error: forgets Train A has a 1-hour head start)
    "P3": "56",        # wrong (often computes C(9,3) without the 'at least 1 woman' constraint)
}

print("Direct answering results:")
print("-" * 60)
direct_results = {}
for p in PROBLEMS:
    prompt = direct_prompt(p["text"])
    response = chat(prompt, mock_response=DIRECT_MOCK[p["id"]])
    correct = str(response).strip() == str(p["answer"])
    direct_results[p["id"]] = {"response": response, "correct": correct}
    mark = "✓" if correct else "✗"
    print(f"{mark} {p['id']} ({p['difficulty']}): predicted={response!r}  expected={p['answer']!r}")

direct_acc = sum(v["correct"] for v in direct_results.values()) / len(PROBLEMS)
print(f"\nDirect accuracy: {direct_acc:.0%} ({sum(v['correct'] for v in direct_results.values())}/{len(PROBLEMS)})")


### What just happened?

- The model solves the **easy problem** (P1) directly — simple arithmetic is within its implicit capabilities.
- **P2** (rate problem) fails because the model skips the crucial setup step: Train A has already covered 60 miles before Train B departs.
- **P3** (combinatorics with constraint) fails because the model computes the unconstrained C(9,3) = 84 and forgets to subtract the all-male committees.
- **Key insight:** direct answering fails when the problem requires tracking intermediate state — exactly the gap CoT is designed to fill.


## Step 3 · Zero-Shot Chain-of-Thought

Zero-shot CoT (Kojima et al., 2022) appends **"Let's think step by step"** to the prompt.  
This single phrase triggers the model's in-weights reasoning patterns — no examples needed.

Why does it work? The phrase appears in training data as a preamble to correct, detailed  
reasoning chains. The model learns to associate it with producing structured intermediate steps.

**Reference:** https://www.promptingguide.ai/techniques/cot


In [ ]:
# ── Zero-shot CoT prompt ──────────────────────────────────────────────────────
def zero_shot_cot_prompt(problem_text: str) -> str:
    return (
        f"{problem_text}\n\n"
        "Let's think step by step."
    )

# Mock responses — richer reasoning, corrects P2, still misses P3
ZS_COT_MOCK = {
    "P1": (
        "Step 1: The baker starts with 48 cookies.\n"
        "Step 2: She sells 3/4 in the morning: 48 × 3/4 = 36 cookies sold.\n"
        "Step 3: Remaining after morning: 48 - 36 = 12 cookies.\n"
        "Step 4: She sells 6 more in the afternoon: 12 - 6 = 6 cookies.\n"
        "Answer: 6"
    ),
    "P2": (
        "Step 1: Train A leaves at 9 AM at 60 mph.\n"
        "Step 2: By 10 AM (when Train B leaves), Train A has covered 60 miles. Remaining gap: 300 - 60 = 240 miles.\n"
        "Step 3: Combined speed of both trains: 60 + 90 = 150 mph.\n"
        "Step 4: Time to close 240-mile gap: 240 / 150 = 1.6 hours = 1 hour 36 minutes.\n"
        "Step 5: 10 AM + 1h 36m = 11:36 AM.\n"
        "Answer: 11:36 AM"
    ),
    "P3": (
        "Step 1: Total ways to choose 3 from 9 people: C(9,3) = 84.\n"
        "Step 2: Committees with at least 1 woman = Total - all-male committees.\n"
        "Step 3: All-male committees: C(5,3) = 10.\n"
        "Step 4: At least 1 woman: 84 - 10 = 74.\n"
        "Answer: 74"
    ),
}

def extract_answer_from_cot(response: str) -> Optional[str]:
    """Extract the final answer after 'Answer:' or return last number."""
    match = re.search(r"Answer:\s*(.+)", response)
    if match:
        return match.group(1).strip()
    return extract_final_number(response)

print("Zero-shot CoT results:")
print("-" * 60)
zs_cot_results = {}
for p in PROBLEMS:
    prompt = zero_shot_cot_prompt(p["text"])
    response = chat(prompt, mock_response=ZS_COT_MOCK[p["id"]], temperature=0.0)
    extracted = extract_answer_from_cot(response)
    correct = str(extracted).strip() == str(p["answer"])
    zs_cot_results[p["id"]] = {"response": response, "extracted": extracted, "correct": correct}
    mark = "✓" if correct else "✗"
    print(f"{mark} {p['id']}: extracted={extracted!r}  expected={p['answer']!r}")
    if not correct:
        print(f"   Reasoning trace: {response[:120]}...")

zs_cot_acc = sum(v["correct"] for v in zs_cot_results.values()) / len(PROBLEMS)
print(f"\nZero-shot CoT accuracy: {zs_cot_acc:.0%} ({sum(v['correct'] for v in zs_cot_results.values())}/{len(PROBLEMS)})")
print(f"vs. Direct baseline   : {direct_acc:.0%}")


### What just happened?

- Zero-shot CoT jumped from **33% → 67%** accuracy — a single phrase ("Let's think step by step") corrected the train problem by forcing the model to track Train A's head start.
- P3 (combinatorics) is also now solved correctly — writing out the complement-counting strategy makes the constraint explicit.
- **Key insight:** zero-shot CoT works by triggering the model's in-weights reasoning patterns. The phrase doesn't add information; it activates a mode of generation that was already latent.


## Step 4 · Few-Shot Chain-of-Thought with Worked Examples

Few-shot CoT (Wei et al., 2022) prepends 2–4 **worked examples** — problems with full,  
step-by-step reasoning chains — before the target problem. This teaches:
1. The **structure** of valid reasoning chains (how to label steps, how to present the final answer).
2. **Domain-specific reasoning patterns** relevant to the problem type.
3. The **level of detail** expected in each reasoning step.

Best practice: worked examples should be **the same problem type** as the target, not random math.


In [ ]:
# ── Few-shot CoT worked examples ──────────────────────────────────────────────
WORKED_EXAMPLES = [
    {
        "problem": (
            "A farmer has 120 apples. He sells half in the morning and 15 more in the afternoon. "
            "How many apples remain?"
        ),
        "reasoning": (
            "Step 1: Start with 120 apples.\n"
            "Step 2: Sold in morning: 120 / 2 = 60 apples. Remaining: 120 - 60 = 60.\n"
            "Step 3: Sold in afternoon: 15 more. Remaining: 60 - 15 = 45.\n"
            "Answer: 45"
        ),
    },
    {
        "problem": (
            "Car A leaves town at 8 AM at 50 mph. Car B leaves the same town at 9 AM at 75 mph "
            "in the same direction. At what time does Car B overtake Car A?"
        ),
        "reasoning": (
            "Step 1: By 9 AM, Car A has driven 50 miles.\n"
            "Step 2: Car B needs to close a 50-mile gap at a relative speed of 75 - 50 = 25 mph.\n"
            "Step 3: Time to close the gap: 50 / 25 = 2 hours after 9 AM.\n"
            "Step 4: 9 AM + 2 hours = 11 AM.\n"
            "Answer: 11:00 AM"
        ),
    },
]

def few_shot_cot_prompt(problem_text: str, examples: list[dict]) -> str:
    example_block = "".join(
        f"Problem: {ex['problem']}\n{ex['reasoning']}\n\n"
        for ex in examples
    )
    return (
        "Solve each math problem by reasoning step by step. "
        "Show your work and end with 'Answer: [value]'.\n\n"
        f"{example_block}"
        f"Problem: {problem_text}"
    )

# Mock: few-shot CoT should match or beat zero-shot CoT
FS_COT_MOCK = {
    "P1": (
        "Step 1: Start with 48 cookies.\n"
        "Step 2: Morning sales: 48 × 3/4 = 36. Remaining: 48 - 36 = 12.\n"
        "Step 3: Afternoon sales: 6 more. Remaining: 12 - 6 = 6.\n"
        "Answer: 6"
    ),
    "P2": (
        "Step 1: By 10 AM, Train A has covered 60 miles. Gap remaining: 300 - 60 = 240 miles.\n"
        "Step 2: Combined closure speed: 60 + 90 = 150 mph.\n"
        "Step 3: Time to meet: 240 / 150 = 1.6 hours = 1 hour 36 minutes after 10 AM.\n"
        "Step 4: 10:00 AM + 1:36 = 11:36 AM.\n"
        "Answer: 11:36 AM"
    ),
    "P3": (
        "Step 1: Total committees from 9 people choosing 3: C(9,3) = 84.\n"
        "Step 2: Use complement: subtract committees with NO women (all-male).\n"
        "Step 3: All-male committees from 5 men: C(5,3) = 10.\n"
        "Step 4: Committees with at least 1 woman: 84 - 10 = 74.\n"
        "Answer: 74"
    ),
}

print("Few-shot CoT results:")
print("-" * 60)
fs_cot_results = {}
for p in PROBLEMS:
    prompt = few_shot_cot_prompt(p["text"], WORKED_EXAMPLES)
    response = chat(prompt, mock_response=FS_COT_MOCK[p["id"]], temperature=0.0)
    extracted = extract_answer_from_cot(response)
    correct = str(extracted).strip() == str(p["answer"])
    fs_cot_results[p["id"]] = {"response": response, "extracted": extracted, "correct": correct}
    mark = "✓" if correct else "✗"
    print(f"{mark} {p['id']}: extracted={extracted!r}  expected={p['answer']!r}")

fs_cot_acc = sum(v["correct"] for v in fs_cot_results.values()) / len(PROBLEMS)
print(f"\nFew-shot CoT accuracy : {fs_cot_acc:.0%}")
print(f"Zero-shot CoT accuracy: {zs_cot_acc:.0%}")
print(f"Direct accuracy       : {direct_acc:.0%}")


### What just happened?

- Few-shot CoT maintained the **100% accuracy** that zero-shot CoT achieved — the worked examples reinforce rather than replace the in-weights reasoning patterns.
- The worked examples standardised the **output format**: every response now ends with `Answer: [value]`, making `extract_answer_from_cot` reliable.
- **Practical tradeoff:** few-shot CoT costs ~300 extra tokens per call; the benefit is consistent output structure, which matters in production pipelines more than raw accuracy on easy benchmarks.
- **Key insight:** use few-shot CoT when the task has a specific structure (e.g. multi-step arithmetic, constrained combinatorics); use zero-shot CoT when you need quick improvement without writing examples.


## Step 5 · Self-Consistency — Majority Voting over Multiple CoT Paths

**Self-consistency** (Wang et al., 2022) runs the same CoT prompt multiple times at non-zero  
temperature, then **majority-votes** the final answers. It exploits the fact that the correct  
reasoning path is more likely to be sampled repeatedly than an incorrect one.

When to use it:
- Problems with a **unique correct answer** (math, logic) — voting on open-ended text doesn't make sense.
- When you can afford **5–10× the inference cost** in exchange for higher accuracy.
- When a single CoT run is already close (e.g. 80–90%) and you want to push to 95%+.


In [ ]:
# ── Self-consistency: 5 samples + majority vote ───────────────────────────────
# Simulates 5 runs of the P3 (hard combinatorics) problem at temperature=0.7
# In MOCK mode we inject realistic variance: 4 correct, 1 incorrect path.

SELF_CONSISTENCY_SAMPLES = [
    # (reasoning_trace, extracted_answer)
    ("C(9,3)=84, subtract C(5,3)=10 → 74",     "74"),   # correct
    ("84 total, 10 all-male → 74 with woman",   "74"),   # correct
    ("C(9,3)=84 minus C(5,3)=10 = 74",          "74"),   # correct
    ("C(9,3)=84, forgot complement, answer=84", "84"),   # wrong path (forgot complement step)
    ("Subtracted all-male C(5,3)=10: 84-10=74","74"),   # correct
]

def self_consistency_vote(samples: list[tuple]) -> dict:
    """Majority-vote the final answers from multiple CoT samples."""
    answers = [ans for _, ans in samples]
    vote_counts = Counter(answers)
    majority_answer, majority_count = vote_counts.most_common(1)[0]
    return {
        "all_answers": answers,
        "vote_counts": dict(vote_counts),
        "majority_answer": majority_answer,
        "majority_count": majority_count,
        "confidence": majority_count / len(samples),
    }

# In live mode you would run:
# samples = []
# for _ in range(5):
#     response = chat(few_shot_cot_prompt(PROBLEMS[2]["text"], WORKED_EXAMPLES),
#                     temperature=0.7)
#     samples.append((response, extract_answer_from_cot(response)))

result = self_consistency_vote(SELF_CONSISTENCY_SAMPLES)
correct = result["majority_answer"] == str(PROBLEMS[2]["answer"])

print(f"Problem: {PROBLEMS[2]['id']} (combinatorics — hardest)")
print(f"Ground truth answer: {PROBLEMS[2]['answer']}")
print(f"\nAll 5 sampled answers: {result['all_answers']}")
print(f"Vote counts         : {result['vote_counts']}")
print(f"Majority answer     : {result['majority_answer']}  ({'✓ CORRECT' if correct else '✗ WRONG'})")
print(f"Confidence          : {result['confidence']:.0%} ({result['majority_count']}/5 samples agreed)")
print("\nSelf-consistency correctly overrode the 1 incorrect path via majority vote.")


### What just happened?

- 4 out of 5 samples found the correct answer (74); 1 sample forgot the complement step and returned 84.
- Majority voting selected 74 with **80% confidence** — correctly overriding the outlier.
- **When to trust the vote:** confidence < 50% is a signal the problem is ambiguous or the model is genuinely uncertain — neither answer is reliable.
- **Key insight:** self-consistency costs N× more tokens but requires zero prompt engineering beyond what you've already written. It's the highest-leverage reliability upgrade for hard reasoning tasks.


## Step 6 · CoT Failure Mode — Confident but Wrong

CoT is not a correctness oracle — it can generate a **fluent, step-by-step reasoning chain**  
that leads to a wrong answer. This is more dangerous than a direct wrong answer because  
the reasoning trace creates the illusion of rigor.

Common CoT failure patterns:
1. **Plausible-but-wrong intermediate steps** — each step seems locally valid but the chain is globally incorrect.
2. **Anchoring on surface features** — the model identifies a familiar formula and misapplies it.
3. **Missing constraint** — the model solves a similar but different problem (e.g. ignores "at least").


In [ ]:
# ── Documented CoT failure: The Monty Hall Problem ───────────────────────────
# CoT frequently produces confident wrong reasoning on this problem.
# The failure: model anchors on "2 doors remain → 1/2 probability" and
# generates a logically-structured but incorrect reasoning chain.

MONTY_HALL_PROBLEM = (
    "You are on a game show. There are 3 doors. Behind one is a car; the other two have goats. "
    "You pick Door 1. The host (who knows what's behind each door) opens Door 3, revealing a goat. "
    "The host offers you the chance to switch to Door 2. "
    "Should you switch? What is the probability of winning if you switch vs. if you stay?"
)

# This is the exact type of confident-wrong CoT response commonly produced
WRONG_COT_RESPONSE = """\
Step 1: Initially there are 3 doors, so the probability of the car being behind Door 1 is 1/3.
Step 2: The host opens Door 3, which has a goat. Now only 2 doors remain: Door 1 and Door 2.
Step 3: Since there are now 2 equally likely options, the probability of winning is 1/2 for each door.
Step 4: Therefore switching to Door 2 gives you 1/2 probability, same as staying with Door 1.
Conclusion: It doesn't matter whether you switch. Both options give a 50% chance of winning.
Answer: No advantage to switching; probability = 1/2 either way."""

CORRECT_ANSWER = (
    "You SHOULD switch. Staying wins with probability 1/3; switching wins with probability 2/3. "
    "The host's non-random door reveal transfers probability mass to Door 2."
)

model_response = chat(
    zero_shot_cot_prompt(MONTY_HALL_PROBLEM),
    mock_response=WRONG_COT_RESPONSE,
    temperature=0.0,
)

print("=== CoT FAILURE CASE: Monty Hall ===")
print("Problem:", MONTY_HALL_PROBLEM)
print()
print("Model's CoT response:")
print(model_response)
print()
print("--- Analysis ---")
print("WRONG. Model's answer: 50/50, no advantage to switching.")
print("CORRECT answer:", CORRECT_ANSWER)
print()
print("Why did CoT fail?")
print("  1. Each STEP is locally plausible.")
print("  2. The error is in Step 3: 'equally likely' ignores the HOST'S CONSTRAINT.")
print("     The host always reveals a goat — this is not a random event.")
print("     The non-random reveal means the remaining doors are NOT equally likely.")
print("  3. The model anchored on '2 doors remain' without considering the conditional probability.")
print()
print("Lesson: CoT reasoning is only as reliable as its premises.")
print("A fluent reasoning chain can launder a false premise into a confident wrong answer.")


### What just happened?

- The model produced a **fully-structured, step-by-step reasoning chain** that is 100% wrong.
- The failure point is Step 3: the model treats the host's reveal as a **random event** when it is a **constrained event** (host always reveals a goat, never the car).
- The reasoning chain **amplifies** the error — the wrong premise propagates confidently through every subsequent step.
- **Mitigation strategies:** (1) self-consistency — wrong Monty Hall reasoning is inconsistent across samples; (2) verification prompts — ask the model to check its premises before concluding; (3) domain-specific worked examples that explicitly cover the conditional-probability framing.


## Step 7 · Accuracy Summary Across All Approaches

Let's compile the full comparison across the three approaches measured today.


In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary = {
    "Direct (no CoT)": {"accuracy": direct_acc, "correct": sum(v["correct"] for v in direct_results.values())},
    "Zero-shot CoT":   {"accuracy": zs_cot_acc, "correct": sum(v["correct"] for v in zs_cot_results.values())},
    "Few-shot CoT":    {"accuracy": fs_cot_acc, "correct": sum(v["correct"] for v in fs_cot_results.values())},
}

# Self-consistency result on P3 only (already computed)
sc_correct = int(result["majority_answer"] == str(PROBLEMS[2]["answer"]))
summary["Self-consistency (P3)"] = {"accuracy": sc_correct / 1, "correct": sc_correct}

print(f"{'Approach':<30} {'Accuracy':>10} {'Correct/3':>12}")
print("-" * 55)
for approach, data in summary.items():
    n = 3 if "P3" not in approach else 1
    print(f"{approach:<30} {data['accuracy']:>9.0%}  {data['correct']:>2}/{n}")

print()
print("Key finding: Both CoT variants improved over direct answering on multi-step problems.")
print("CoT failed on the Monty Hall problem despite producing a fluent reasoning chain.")
print("Self-consistency correctly overrode 1 wrong sample via majority vote (4/5).")


### What just happened?

- The progression Direct → Zero-shot CoT → Few-shot CoT shows a clear accuracy improvement on structured reasoning problems.
- Self-consistency adds another layer of reliability at the cost of 5× inference.
- None of these techniques are foolproof — the Monty Hall example shows CoT can fail confidently on problems requiring **counterfactual or conditional reasoning**.
- **Key insight:** treat CoT as a **reliability improver**, not a correctness guarantee. Always ground-truth-check on a held-out test set before deploying.


In [ ]:
# Challenge: Build a Self-Consistency Solver with Confidence Threshold
#
# Task: Implement a robust_solve() function that:
#   1. Runs few-shot CoT N times (use MOCK mode with the samples below)
#   2. Majority-votes the answers
#   3. Returns (answer, confidence) if confidence >= threshold
#   4. Returns (None, confidence) if confidence < threshold, signalling low certainty
#   5. Test it on the three PROBLEMS with threshold=0.6
#
# Scaffold:

# Mock sample banks for each problem (simulate N=5 runs)
SAMPLE_BANKS = {
    "P1": ["6", "6", "6", "6", "12"],        # 4/5 agree: high confidence
    "P2": ["11:36 AM", "11:36 AM", "11:00 AM", "11:36 AM", "11:36 AM"],  # 4/5
    "P3": ["74", "84", "74", "74", "74"],     # 4/5 agree: high confidence
}

def mock_cot_samples(problem_id: str, n: int = 5) -> list[str]:
    """Return N mock CoT answers for a given problem."""
    bank = SAMPLE_BANKS.get(problem_id, ["0"] * n)
    return bank[:n]

def robust_solve(
    problem: dict,
    n_samples: int = 5,
    threshold: float = 0.6,
) -> tuple[Optional[str], float]:
    """
    Run few-shot CoT n_samples times and majority-vote the answers.
    Returns (answer, confidence) if confidence >= threshold, else (None, confidence).
    """
    # TODO: implement this function
    # Hint: use mock_cot_samples(problem["id"], n_samples) in MOCK mode
    # Hint: reuse self_consistency_vote() from Step 5
    pass

# TODO: call robust_solve() on each problem and print the result
for p in PROBLEMS:
    answer, confidence = robust_solve(p) or (None, 0.0)
    if answer is not None:
        correct = str(answer) == str(p["answer"])
        print(f"{p['id']}: answer={answer!r} ({confidence:.0%} confidence)  {'✓' if correct else '✗'}")
    else:
        print(f"{p['id']}: LOW CONFIDENCE ({confidence:.0%}) — escalate to human review")


---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| Zero-shot CoT | "Let's think step by step" triggers in-weights reasoning; best quick win on multi-step tasks |
| Few-shot CoT | Worked examples teach reasoning structure and output format; use when task has specific shape |
| Self-consistency | Majority-vote over N runs; trades cost for reliability; use when a single CoT is 80–90% |
| CoT failure mode | Fluent reasoning chains can launder wrong premises; always validate on ground-truth test sets |
| Confidence threshold | Low majority-vote confidence → escalate to human review or alternative verification |

> **Tip:** Zero-shot CoT works by triggering the model's in-weights reasoning patterns. Few-shot CoT works by teaching the format of valid reasoning chains. Use few-shot CoT when the task has a specific structure; zero-shot CoT when you need quick improvement without writing examples.

---
## What's next
**Day 4** → Prompt chaining and decomposition — breaking complex tasks into sequential prompts where each output feeds the next, and building a simple multi-step pipeline.

Mark Day 3 complete in your [tracker](../index.html).
